# 02b — Check: why was GPTQ (C4) worse than RTN on Qwen?

Notebook 02 gave WikiText2 perplexity: FP16 9.65 | RTN-4 10.74 | GPTQ-4 C4 11.18.
On OPT-125M GPTQ matched the paper, so the algorithm works. This notebook runs three checks:

1. **C4 perplexity** of FP16 / RTN / GPTQ (the saved models). If GPTQ is better than RTN on C4
   (its calibration text) but worse on WikiText2, GPTQ is over-fitting its calibration data.
2. **Layer-level check:** for some layers, the output error ||WX − ŴX||² of RTN vs GPTQ,
   on calibration (C4) inputs and on new (WikiText2) inputs.
3. **Standard GPTQ options** used by most real GPTQ releases:
   - **act-order**: quantize the most important input columns first;
   - **static groups**: group scales fixed from the original weights, so GPTQ uses
     *exactly the same grid as RTN* (only the rounding decisions differ);
   - **true-sequential**: inside each block quantize q/k/v → o → gate/up → down, one after another.

**Before running:** + Add Input → Your Work → **02-Quantize** (to load the saved RTN and GPTQ models).
Then **Save Version → Save & Run All**. GPU T4 x2, Internet On. About 50–70 min.
Every finished step is saved (checkpointing, same as notebook 02).

In [ ]:
# 1. Versions and GPU
import sys, torch, transformers, datasets
print("torch", torch.__version__, "| transformers", transformers.__version__,
      "| GPU", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - turn on the GPU!")

In [ ]:
# 2. Settings (same as notebook 02)
QWEN_ID  = "Qwen/Qwen2.5-1.5B-Instruct"
BITS, GROUP, NSAMPLES, SEQLEN, SEED, DAMP = 4, 128, 128, 2048, 0, 0.01
OUT_DIR  = "/kaggle/working"
RTN_NAME  = "qwen2.5-1.5b-rtn4-g128"
GPTQ_NAME = "qwen2.5-1.5b-gptq4-g128-c4"

In [ ]:
# 3. RTN and GPTQ code (now with act-order, static groups and true-sequential options)
# RTN and GPTQ in plain PyTorch, following the original GPTQ code
# (IST-DASLab/gptq: quant.py, gptq.py, opt.py). Both methods use the SAME
# Quantizer, so they share exactly the same grid definition.
import copy
import math
import torch
import torch.nn as nn


# ---------------------------------------------------------------------------
# 1. The grid (asymmetric min-max, as in Section 2.1 of the survey)
# ---------------------------------------------------------------------------
def quantize(x, scale, zero, maxq):
    """q = clip(round(x/s) + z, 0, 2^b - 1);  w_hat = s * (q - z)"""
    q = torch.clamp(torch.round(x / scale) + zero, 0, maxq)
    return scale * (q - zero)


class Quantizer:
    def __init__(self, bits, sym=False):
        self.maxq = 2 ** bits - 1
        self.sym = sym

    def find_params(self, w):
        """One scale and zero-point per row of w (w = one group of columns)."""
        xmin = torch.minimum(w.min(dim=1).values, torch.zeros(1, device=w.device))
        xmax = torch.maximum(w.max(dim=1).values, torch.zeros(1, device=w.device))
        if self.sym:
            xmax = torch.maximum(xmin.abs(), xmax)
            xmin = -xmax
        both_zero = (xmin == 0) & (xmax == 0)
        xmin[both_zero], xmax[both_zero] = -1, 1
        self.scale = ((xmax - xmin) / self.maxq).unsqueeze(1)
        if self.sym:
            self.zero = torch.full_like(self.scale, (self.maxq + 1) / 2)
        else:
            self.zero = torch.round(-xmin / self.scale.squeeze(1)).unsqueeze(1)

    def __call__(self, x):
        return quantize(x, self.scale, self.zero, self.maxq)


# ---------------------------------------------------------------------------
# 2. RTN: round each group independently, no data
# ---------------------------------------------------------------------------
@torch.no_grad()
def rtn_linear_(layer, bits, groupsize, sym=False):
    W = layer.weight.data.float()
    cols = W.shape[1]
    g = cols if groupsize == -1 else groupsize
    qz = Quantizer(bits, sym)
    Q = torch.empty_like(W)
    for c in range(0, cols, g):
        qz.find_params(W[:, c:c + g])
        Q[:, c:c + g] = qz(W[:, c:c + g])
    layer.weight.data = Q.to(layer.weight.dtype)


# ---------------------------------------------------------------------------
# 3. GPTQ for one linear layer (Algorithm 1 of the paper)
# ---------------------------------------------------------------------------
class GPTQ:
    def __init__(self, layer):
        self.layer = layer
        cols = layer.weight.shape[1]
        self.H = torch.zeros((cols, cols), device=layer.weight.device, dtype=torch.float32)
        self.nsamples = 0

    def add_batch(self, inp):
        """Accumulate H = 2 X X^T / n from the inputs this layer receives."""
        inp = inp.reshape(-1, inp.shape[-1]).t().float()        # (cols, tokens)
        self.H *= self.nsamples / (self.nsamples + 1)
        self.nsamples += 1
        inp = math.sqrt(2 / self.nsamples) * inp
        self.H += inp @ inp.t()

    @torch.no_grad()
    def quantize_(self, bits, groupsize=128, sym=False, percdamp=0.01, blocksize=128,
                  act_order=False, static_groups=False):
        """act_order: quantize columns with the largest H diagonal first ("desc_act").
        static_groups: each group's scale/zero-point is fixed from the ORIGINAL weights,
        so GPTQ uses exactly the same grid as RTN (only rounding decisions differ)."""
        W = self.layer.weight.data.clone().float()
        rows, cols = W.shape
        qz = Quantizer(bits, sym)
        if groupsize == -1:
            qz.find_params(W)
        H = self.H
        del self.H
        dead = torch.diag(H) == 0                    # inputs that are always 0
        H[dead, dead] = 1
        W[:, dead] = 0
        groups = None
        if groupsize != -1 and (static_groups or act_order):
            groups = []
            for c in range(0, cols, groupsize):
                g = Quantizer(bits, sym)
                g.find_params(W[:, c:c + groupsize])
                groups.append(g)
        perm = None
        if act_order:
            perm = torch.argsort(torch.diag(H), descending=True)
            W = W[:, perm]
            H = H[perm][:, perm]
            invperm = torch.argsort(perm)
        damp = percdamp * torch.mean(torch.diag(H))  # 1% dampening
        H += damp * torch.eye(cols, device=H.device)
        # Cholesky form of H^-1 (upper triangular), as in the paper
        H = torch.linalg.cholesky(H)
        H = torch.cholesky_inverse(H)
        Hinv = torch.linalg.cholesky(H, upper=True)

        Q = torch.zeros_like(W)
        losses = torch.zeros(rows, device=W.device)
        for i1 in range(0, cols, blocksize):          # lazy batch updates
            i2 = min(i1 + blocksize, cols)
            W1 = W[:, i1:i2].clone()
            Q1 = torch.zeros_like(W1)
            Err1 = torch.zeros_like(W1)
            Hinv1 = Hinv[i1:i2, i1:i2]
            for i in range(i2 - i1):                  # fixed column order
                if groups is not None:
                    col = int(perm[i1 + i]) if perm is not None else i1 + i
                    qz = groups[col // groupsize]
                elif groupsize != -1 and (i1 + i) % groupsize == 0:
                    qz.find_params(W[:, i1 + i:i1 + i + groupsize])
                w = W1[:, i]
                d = Hinv1[i, i]
                q = qz(w.unsqueeze(1)).flatten()
                Q1[:, i] = q
                losses += (w - q) ** 2 / d ** 2 / 2
                err = (w - q) / d
                W1[:, i:] -= err.unsqueeze(1) @ Hinv1[i, i:].unsqueeze(0)
                Err1[:, i] = err
            Q[:, i1:i2] = Q1
            W[:, i2:] -= Err1 @ Hinv[i1:i2, i2:]
        if perm is not None:
            Q = Q[:, invperm]
        self.layer.weight.data = Q.to(self.layer.weight.dtype)
        return losses.sum().item()


# ---------------------------------------------------------------------------
# 4. Whole-model helpers
# ---------------------------------------------------------------------------
def get_layers(model):
    """Transformer blocks for Qwen/Llama-style and OPT-style models."""
    inner = model.model
    if hasattr(inner, "layers"):
        return inner.layers
    if hasattr(inner, "decoder") and hasattr(inner.decoder, "layers"):
        return inner.decoder.layers
    raise ValueError("Unknown architecture")


def block_linears(block):
    return {n: m for n, m in block.named_modules() if isinstance(m, nn.Linear)}


@torch.no_grad()
def rtn_model_(model, bits=4, groupsize=128, sym=False):
    for block in get_layers(model):
        for m in block_linears(block).values():
            rtn_linear_(m, bits, groupsize, sym)


class _Catcher(nn.Module):
    """Stands in for block 0 to record its inputs, then stops the forward pass."""
    def __init__(self, m, store):
        super().__init__()
        self.m = m
        self.store = store

    def __getattr__(self, name):          # pass through attributes like attention_type
        try:
            return super().__getattr__(name)
        except AttributeError:
            return getattr(self.m, name)

    def forward(self, *args, **kwargs):
        hs = args[0] if args else kwargs.pop("hidden_states")
        self.store["inps"].append(hs)
        if self.store["kwargs"] is None:
            self.store["kwargs"] = kwargs
        raise ValueError("captured")


def _first(out):
    return out[0] if isinstance(out, (tuple, list)) else out


# Order used by "true sequential" GPTQ (Llama/Qwen-style blocks)
TRUE_SEQ = [["self_attn.q_proj", "self_attn.k_proj", "self_attn.v_proj"],
            ["self_attn.o_proj"], ["mlp.gate_proj", "mlp.up_proj"], ["mlp.down_proj"]]


@torch.no_grad()
def gptq_model_(model, calib, bits=4, groupsize=128, sym=False,
                percdamp=0.01, blocksize=128, verbose=True,
                act_order=False, static_groups=False, true_sequential=False):
    """Quantize block by block. Each block is calibrated on the outputs of the
    blocks before it that are ALREADY quantized (as in the GPTQ paper)."""
    layers = get_layers(model)
    dev = next(model.parameters()).device
    store = {"inps": [], "kwargs": None}
    layers[0] = _Catcher(layers[0], store)
    for x in calib:
        try:
            model(x.to(dev), use_cache=False)
        except ValueError:
            pass
    layers[0] = layers[0].m
    inps = torch.cat(store["inps"], dim=0)
    kw = store["kwargs"]
    outs = torch.zeros_like(inps)

    for li, block in enumerate(layers):
        lins = block_linears(block)
        names = list(lins)
        if true_sequential and all(n in lins for grp in TRUE_SEQ for n in grp) \
                and len(names) == sum(len(g) for g in TRUE_SEQ):
            steps = TRUE_SEQ     # quantize q/k/v, then o, then gate/up, then down
        else:
            steps = [names]      # all layers of the block at once (original GPTQ code)
        err = {}
        for step in steps:
            gptq = {n: GPTQ(lins[n]) for n in step}
            hooks = [lins[n].register_forward_hook(
                        lambda mod, inp, out, n=n: gptq[n].add_batch(inp[0].data))
                     for n in step]
            for j in range(inps.shape[0]):
                block(inps[j:j + 1], **kw)
            for h in hooks:
                h.remove()
            for n in step:
                err[n] = gptq[n].quantize_(bits, groupsize, sym, percdamp, blocksize,
                                           act_order, static_groups)
                del gptq[n]
        for j in range(inps.shape[0]):           # outputs of the QUANTIZED block
            outs[j] = _first(block(inps[j:j + 1], **kw))[0]
        inps, outs = outs, inps
        if verbose:
            print(f"  block {li:2d} done | total error {sum(err.values()):.1f}")
        torch.cuda.empty_cache() if torch.cuda.is_available() else None


# ---------------------------------------------------------------------------
# 5. Check: every (row, group) uses at most 2^bits distinct values
# ---------------------------------------------------------------------------
@torch.no_grad()
def check_grid(model, bits, groupsize):
    worst = 0
    for block in get_layers(model):
        for m in block_linears(block).values():
            W = m.weight.data.float()
            g = W.shape[1] if groupsize == -1 else groupsize
            for c in range(0, W.shape[1], g):
                grp = W[:, c:c + g]
                worst = max(worst, max(len(torch.unique(r)) for r in grp[:8]))
    return worst

In [ ]:
# 4. Data, perplexity, checkpointing
import os, glob, shutil, random, gc, json, time
import torch.nn.functional as F
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

c4_train = load_dataset("allenai/c4", data_files={"train": "en/c4-train.00000-of-01024.json.gz"},
                        split="train")
c4_val = load_dataset("allenai/c4", data_files={"validation": "en/c4-validation.00000-of-00008.json.gz"},
                      split="validation")
wiki_text = "\n\n".join(load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="test")["text"])
tok = AutoTokenizer.from_pretrained(QWEN_ID)

def get_c4_calib(n=NSAMPLES, seqlen=SEQLEN, seed=SEED):     # identical to notebook 02
    random.seed(seed)
    out = []
    while len(out) < n:
        ids = tok(c4_train[random.randint(0, len(c4_train) - 1)]["text"], return_tensors="pt").input_ids
        if ids.shape[1] <= seqlen:
            continue
        i = random.randint(0, ids.shape[1] - seqlen - 1)
        out.append(ids[:, i:i + seqlen])
    return out

WIKI_IDS = tok(wiki_text, return_tensors="pt").input_ids
# C4 validation text as in the GPTQ code: first 1,100 documents, 256 x 2,048 tokens
C4_IDS = tok(" ".join(c4_val[:1100]["text"]), return_tensors="pt").input_ids[:, :256 * SEQLEN]

@torch.no_grad()
def ppl(model, ids, seqlen=SEQLEN):
    n = ids.shape[1] // seqlen
    nll = 0.0
    for i in range(n):
        x = ids[:, i * seqlen:(i + 1) * seqlen].to(model.device)
        logits = model(x, use_cache=False).logits[:, :-1, :].float()
        nll += F.cross_entropy(logits.reshape(-1, logits.shape[-1]), x[:, 1:].reshape(-1)).item() * seqlen
    return math.exp(nll / (n * seqlen))

def load(path):
    return AutoModelForCausalLM.from_pretrained(path, dtype=torch.float16, device_map={"": 0}).eval()

def free():
    gc.collect(); torch.cuda.empty_cache()

def find_input(name):
    hits = [h for h in glob.glob(f"/kaggle/input/**/{name}", recursive=True) if os.path.isdir(h)]
    if not hits:
        raise FileNotFoundError(f"Add notebook 02's output as an Input ('{name}' not found)")
    return hits[0]

def restore(name):
    dst = os.path.join(OUT_DIR, name)
    if os.path.exists(dst):
        return True
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)   # names are unique to 02b
    if not hits:
        return False
    (shutil.copytree if os.path.isdir(hits[0]) else shutil.copy)(hits[0], dst)
    print("restored", name)
    return True

RES = "02b_results.json"
res = json.load(open(os.path.join(OUT_DIR, RES))) if restore(RES) else {}
def save():
    json.dump(res, open(os.path.join(OUT_DIR, RES), "w"), indent=1)
print("already done:", list(res))

## Check 1 — perplexity on WikiText2 *and* C4 for the saved models

In [ ]:
# 5. FP16 / RTN / GPTQ-C4 on WikiText2 and C4
for tag, path in [("fp16", QWEN_ID), ("rtn4", find_input(RTN_NAME)), ("gptq4", find_input(GPTQ_NAME))]:
    if f"{tag}_c4" in res:
        continue
    m = load(path)
    res[f"{tag}_wiki"] = ppl(m, WIKI_IDS)
    res[f"{tag}_c4"] = ppl(m, C4_IDS)
    del m; free(); save()
print(f"{'model':8s}{'WikiText2':>11s}{'C4':>9s}")
for tag in ["fp16", "rtn4", "gptq4"]:
    print(f"{tag:8s}{res[tag + '_wiki']:11.3f}{res[tag + '_c4']:9.3f}")

## Check 2 — layer-level error: does GPTQ beat RTN layer by layer?

In [ ]:
# 6. For a few layers: error ||(W - W_hat) X||^2 of RTN and GPTQ variants,
#    measured on calibration (C4) inputs and on new (WikiText2) inputs.
#    Uses H = X X^T, since ||(W - W_hat) X||^2 = trace(D H D^T) with D = W - W_hat.
if "layer_check" not in res:
    m = load(QWEN_ID)
    layers = get_layers(m)
    pick = [(b, n) for b in (1, 5, 14, 27) for n in ("self_attn.q_proj", "self_attn.o_proj", "mlp.down_proj")]
    mods = {f"{b}.{n}": dict(layers[b].named_modules())[n] for b, n in pick}
    H = {k: {"c4": GPTQ(mod), "wiki": GPTQ(mod)} for k, mod in mods.items()}
    def collect(src, samples):
        hooks = [mod.register_forward_hook(lambda md, inp, out, k=k: H[k][src].add_batch(inp[0].data))
                 for k, mod in mods.items()]
        with torch.no_grad():
            for x in samples:
                m(x.to(m.device), use_cache=False)
        for h in hooks:
            h.remove()
    calib = get_c4_calib()[:32]
    collect("c4", calib)
    collect("wiki", [WIKI_IDS[:, i * SEQLEN:(i + 1) * SEQLEN] for i in range(16)])
    rows = []
    for k, mod in mods.items():
        Hc, Hw = H[k]["c4"].H, H[k]["wiki"].H
        W0 = mod.weight.data.clone()
        def err(Wq, Hm):
            D = (W0.float() - Wq.float())
            return (D @ Hm * D).sum().item()
        out = {"layer": k}
        tmp = torch.nn.Linear(W0.shape[1], W0.shape[0], bias=False).to(W0.device, W0.dtype)
        tmp.weight.data = W0.clone(); rtn_linear_(tmp, BITS, GROUP); Wr = tmp.weight.data.clone()
        variants = {"gptq": {}, "gptq_act": dict(act_order=True, static_groups=True)}
        Wg = {}
        for v, kw in variants.items():
            tmp.weight.data = W0.clone()
            g = GPTQ(tmp); g.H = Hc.clone()
            g.quantize_(BITS, GROUP, percdamp=DAMP, **kw)
            Wg[v] = tmp.weight.data.clone()
        for src, Hm in [("c4", Hc), ("wiki", Hw)]:
            e_rtn = err(Wr, Hm)
            for v in variants:
                out[f"{v}/rtn_{src}"] = err(Wg[v], Hm) / e_rtn
        rows.append(out)
        del tmp
    res["layer_check"] = rows
    del m, H; free(); save()

print("Error of GPTQ divided by error of RTN  (< 1 means GPTQ is better)")
print(f"{'layer':22s}{'GPTQ C4':>9s}{'GPTQ wiki':>11s}{'act C4':>9s}{'act wiki':>10s}")
for r in res["layer_check"]:
    print(f"{r['layer']:22s}{r['gptq/rtn_c4']:9.3f}{r['gptq/rtn_wiki']:11.3f}"
          f"{r['gptq_act/rtn_c4']:9.3f}{r['gptq_act/rtn_wiki']:10.3f}")

## Check 3 — GPTQ with the standard options (act-order + static groups, then + true-sequential)

In [ ]:
# 7. GPTQ-4bit C4 with act-order + static groups  (~15 min)
ACT_NAME = "qwen2.5-1.5b-gptq4-g128-c4-act"
if "gptq4_act_c4" not in res or not restore(ACT_NAME):
    t0 = time.time()
    m = load(QWEN_ID)
    gptq_model_(m, get_c4_calib(), BITS, GROUP, percdamp=DAMP, verbose=False,
                act_order=True, static_groups=True)
    print(f"max distinct values per group: {check_grid(m, BITS, GROUP)} (must be <= {2**BITS})")
    m.save_pretrained(f"{OUT_DIR}/{ACT_NAME}"); tok.save_pretrained(f"{OUT_DIR}/{ACT_NAME}")
    res["gptq4_act_wiki"] = ppl(m, WIKI_IDS)
    res["gptq4_act_c4"] = ppl(m, C4_IDS)
    del m; free(); save()
    print(f"({time.time() - t0:.0f} s)")
print(f"GPTQ act-order:  WikiText2 {res['gptq4_act_wiki']:.3f} | C4 {res['gptq4_act_c4']:.3f}")

In [ ]:
# 8. GPTQ-4bit C4 with act-order + static groups + true-sequential  (~20 min)
TS_NAME = "qwen2.5-1.5b-gptq4-g128-c4-act-ts"
if "gptq4_actts_c4" not in res or not restore(TS_NAME):
    t0 = time.time()
    m = load(QWEN_ID)
    gptq_model_(m, get_c4_calib(), BITS, GROUP, percdamp=DAMP, verbose=False,
                act_order=True, static_groups=True, true_sequential=True)
    print(f"max distinct values per group: {check_grid(m, BITS, GROUP)} (must be <= {2**BITS})")
    m.save_pretrained(f"{OUT_DIR}/{TS_NAME}"); tok.save_pretrained(f"{OUT_DIR}/{TS_NAME}")
    res["gptq4_actts_wiki"] = ppl(m, WIKI_IDS)
    res["gptq4_actts_c4"] = ppl(m, C4_IDS)
    del m; free(); save()
    print(f"({time.time() - t0:.0f} s)")
print(f"GPTQ act-order + true-seq:  WikiText2 {res['gptq4_actts_wiki']:.3f} | C4 {res['gptq4_actts_c4']:.3f}")

In [ ]:
# 9. Summary
print(f"{'model':28s}{'WikiText2':>10s}{'C4':>9s}")
for name, key in [("FP16", "fp16"), ("RTN-4", "rtn4"), ("GPTQ-4 (notebook 02)", "gptq4"),
                  ("GPTQ-4 act-order", "gptq4_act"), ("GPTQ-4 act-order + true-seq", "gptq4_actts")]:
    print(f"{name:28s}{res[key + '_wiki']:10.3f}{res[key + '_c4']:9.3f}")
best = min(["gptq4", "gptq4_act", "gptq4_actts"], key=lambda k: res[k + "_wiki"] + res[k + "_c4"])
print("\nbest GPTQ variant (WikiText2 + C4):", best)
print("beats RTN on both?", res[best + "_wiki"] < res["rtn4_wiki"] and res[best + "_c4"] < res["rtn4_c4"])